In [1]:
import os
import sys
import json
import hashlib
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

# TODO: Add these imports
# HINT: import torch
# HINT: import torch.nn.functional as F
# HINT: from transformers import AutoModel, AutoTokenizer
# HINT: from huggingface_hub import snapshot_download, HfApi

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import snapshot_download, HfApi

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

print("Working directory:", os.getcwd())
print("Python:", sys.version)

e:\conda\envs\melika-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: f:\git\MLOps-Assignments\HW03\HW3_Student\HW3_Student\HW3_A
Python: 3.10.14 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:44:50) [MSC v.1916 64 bit (AMD64)]


# Step 1: Download model from HuggingFace

Download `sentence-transformers/all-MiniLM-L6-v2` and save 6 files to `bundle/model/`:

| # | File |
|---|------|
| 1 | `config.json` |
| 2 | `tokenizer_config.json` |
| 3 | `tokenizer.json` |
| 4 | `vocab.txt` |
| 5 | `special_tokens_map.json` |
| 6 | `model.safetensors` |

Use `snapshot_download` from `huggingface_hub` or download manually with `AutoModel.save_pretrained()` + `AutoTokenizer.save_pretrained()`.

After downloading, save the git commit hash to `bundle/model/.commit`.

In [2]:
# TODO: Download the 6 model files to bundle/model/

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
BUNDLE_MODEL_DIR = Path("bundle/model")
BUNDLE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# HINT: Use snapshot_download(repo_id=MODEL_ID, revision="main",
#        local_dir=str(BUNDLE_MODEL_DIR), allow_patterns=[...])
# HINT: Required files: config.json, tokenizer_config.json, tokenizer.json,
#        vocab.txt, special_tokens_map.json, model.safetensors

# --- YOUR CODE HERE ---
REQUIRED = ["config.json", "tokenizer_config.json", "tokenizer.json",
            "vocab.txt", "special_tokens_map.json", "model.safetensors"]

snapshot_download(
    repo_id=MODEL_ID,
    revision="main",
    local_dir=str(BUNDLE_MODEL_DIR),
    allow_patterns=REQUIRED,
)

# --- END YOUR CODE ---

# Save commit hash
# HINT: from huggingface_hub import HfApi
# HINT: commit = HfApi().model_info(MODEL_ID, revision="main").sha
# --- YOUR CODE HERE ---

commit = HfApi().model_info(MODEL_ID, revision="main").sha
(BUNDLE_MODEL_DIR / ".commit").write_text(commit)
print("Resolved commit:", commit)

# --- END YOUR CODE ---

# Verify all 6 files exist
for fname in REQUIRED:
    fpath = BUNDLE_MODEL_DIR / fname
    assert fpath.exists(), f"MISSING: {fname}"
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  [OK] {fname:35s} {size_mb:7.1f} MB")

Fetching 6 files: 100%|██████████| 6/6 [00:26<00:00,  4.40s/it]


Resolved commit: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
  [OK] config.json                             0.0 MB
  [OK] tokenizer_config.json                   0.0 MB
  [OK] tokenizer.json                          0.4 MB
  [OK] vocab.txt                               0.2 MB
  [OK] special_tokens_map.json                 0.0 MB
  [OK] model.safetensors                      86.7 MB


# Step 2: Write metadata.json

Create `bundle/metadata.json` with accurate information about the bundle.
Required fields:
- `model_name`: the HuggingFace model ID
- `model_revision`: the git commit hash from the `.commit` file
- `embedding_dim`: 384
- `max_seq_len`: 256
- `framework_version`: the installed torch version
- `transformers_version`: the installed transformers version
- `built_by`: YOUR NAME
- `build_timestamp_utc`: current time in ISO 8601 UTC format

In [3]:
# TODO: Create bundle/metadata.json

# HINT: commit = (BUNDLE_MODEL_DIR / ".commit").read_text().strip()
# HINT: ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
# HINT: torch_version = torch.__version__
# HINT: transformers_version = importlib.metadata.version("transformers")

# --- YOUR CODE HERE ---

import importlib

commit = (BUNDLE_MODEL_DIR / ".commit").read_text().strip()
ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

metadata = {
    "model_name": MODEL_ID,
    "model_revision": commit,
    "embedding_dim": 384,
    "max_seq_len": 256,
    "framework_version": torch.__version__,
    "transformers_version": importlib.metadata.version("transformers"),
    "built_by": "Melika Nobakhtian",        
    "build_timestamp_utc": ts,
}

# --- END YOUR CODE ---

# Write to file
meta_path = Path("bundle/metadata.json")
meta_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metadata, indent=2))

{
  "model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "model_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "embedding_dim": 384,
  "max_seq_len": 256,
  "framework_version": "2.11.0+cpu",
  "transformers_version": "5.6.2",
  "built_by": "Melika Nobakhtian",
  "build_timestamp_utc": "2026-06-19T11:55:08Z"
}


# Step 3: Write MANIFEST.json

Compute SHA-256 hash for every file under `bundle/` (except MANIFEST.json itself)
and write the manifest to `bundle/MANIFEST.json`.

The format must be:
```json
{
  "format_version": 1,
  "files": {
    "relative/path/to/file": "sha256hexdigest",
    ...
  }
}
```

Pro tip: you can peek at `scripts/gen_manifest.py` for reference.

In [4]:
# TODO: Hash every file in bundle/ and create MANIFEST.json

# HINT: def sha256(filepath):
# HINT:     h = hashlib.sha256()
# HINT:     with open(filepath, "rb") as f:
# HINT:         for chunk in iter(lambda: f.read(1 << 20), b""):
# HINT:             h.update(chunk)
# HINT:     return h.hexdigest()

# --- YOUR CODE HERE ---

def sha256(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# --- END YOUR CODE ---

# Build and write manifest
# HINT: root = Path("bundle")
# HINT: for p in sorted(root.rglob("*")):
# HINT:     if p.is_file() and p.name != "MANIFEST.json":
# HINT:         rel = str(p.relative_to(root))
# HINT:         files[rel] = sha256(p)
# --- YOUR CODE HERE ---
root = Path("bundle")
files = {}
for p in sorted(root.rglob("*")):
    if p.is_file() and p.name != "MANIFEST.json":
        files[str(p.relative_to(root))] = sha256(p)

manifest = {"format_version": 1, "files": files}

# --- END YOUR CODE ---

# Write
manifest_path = Path("bundle/MANIFEST.json")
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"MANIFEST.json written with {len(manifest['files'])} files")
for rel, h in sorted(manifest["files"].items()):
    print(f"  {h[:16]}...  {rel}")

MANIFEST.json written with 19 files
  fe6fb7e1ed132cec...  metadata.json
  684888c0ebb17f37...  model\.cache\huggingface\.gitignore
  e8893e2dcb3f4545...  model\.cache\huggingface\CACHEDIR.TAG
  9fb1110d51a410bc...  model\.cache\huggingface\download\config.json.metadata
  3a577d5e7f32f992...  model\.cache\huggingface\download\model.safetensors.metadata
  fdaf36b532935b3d...  model\.cache\huggingface\download\special_tokens_map.json.metadata
  8eac964e51459d2c...  model\.cache\huggingface\download\tokenizer.json.metadata
  c2ac44b17759c44e...  model\.cache\huggingface\download\tokenizer_config.json.metadata
  2f14cb4b1c30aef7...  model\.cache\huggingface\download\vocab.txt.metadata
  d36d67e02523717e...  model\.commit
  e3b0c44298fc1c14...  model\.gitkeep
  953f9c0d463486b1...  model\config.json
  53aa51172d142c89...  model\model.safetensors
  303df45a03609e4e...  model\special_tokens_map.json
  be50c3628f2bf5bb...  model\tokenizer.json
  acb92769e8195aab...  model\tokenizer_config.json

# Step 4: Write predict.py

Write `bundle/predict.py` with exactly 4 functions:

| Function | Signature | Returns |
|----------|-----------|---------|
| `load_bundle()` | no args | `(model, tokenizer)` tuple |
| `embed(texts)` | `List[str]` | `np.ndarray` shape `(N, 384)` float32 |
| `similarity(a, b)` | two `np.ndarray` | `float` (cosine similarity) |
| `info()` | no args | `dict` with metadata |

The **7-step pipeline** inside `embed()`:
1. Tokenize: `tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")`
2. Move tensors to device (cpu or cuda)
3. Forward pass under `torch.no_grad()` → `last_hidden_state`
4. Mean-pool: `sum(H * mask) / sum(mask).clamp(min=1e-9)`
5. L2 normalize: `F.normalize(pooled, p=2, dim=1)`
6. Detach, move to CPU, convert to `np.float32`
7. Return ndarray

**Important rules:**
- DO NOT import `sentence-transformers`. Use raw `transformers` only.
- Set `torch.manual_seed(0)` for determinism.
- Call `model.eval()` before inference.
- A template already exists at `bundle/predict.py` — open it and fill in the TODOs.

In [5]:
# TODO: Write the implementation to bundle/predict.py
#
# Open bundle/predict.py in your editor and fill in the 4 functions:
#   load_bundle(), embed(), similarity(), info()
#
# Then run this cell to verify the module can be imported:

import sys
sys.path.insert(0, "bundle")

# HINT: After writing predict.py, uncomment these lines to test:
from predict import load_bundle, embed, similarity, info
model, tokenizer = load_bundle()
print("Model loaded:", type(model).__name__)
print("Tokenizer loaded:", type(tokenizer).__name__)
print(info())

print("Edit bundle/predict.py, then re-run this cell to verify imports.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3193.10it/s]


Model loaded: BertModel
Tokenizer loaded: BertTokenizer
{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dim': 384, 'max_seq_len': 256, 'device': 'cpu', 'framework': 'transformers', 'deterministic': True, 'bundle_dir': 'f:\\git\\MLOps-Assignments\\HW03\\HW3_Student\\HW3_Student\\HW3_A\\bundle\\model'}
Edit bundle/predict.py, then re-run this cell to verify imports.


# Step 5: Run tests

Run all 4 test files. All tests must pass with green dots.

Tests check:
- **test_parity.py** (7 tests): Correct embedding shape, L2 normalization, similarity
- **test_tokenization.py** (5 tests): Tokenizer behavior, special tokens, round-trip
- **test_determinism.py** (1 test): Same input → same output every time
- **test_adversarial.py** (10 tests): Edge cases (unicode, numerics, long text, empty strings)

In [11]:
# Run all tests
#!cd HW3_A && PYTHONPATH=bundle python -m pytest tests/ -v --tb=short

# Or if you're already in the HW3_A directory:
#!PYTHONPATH=bundle python -m pytest tests/ -v --tb=short
%env PYTHONPATH=bundle
!python -m pytest tests/ -v --tb=short

env: PYTHONPATH=bundle
============================= test session starts =============================
platform win32 -- Python 3.10.14, pytest-9.1.1, pluggy-1.6.0 -- e:\conda\envs\melika-env\python.exe
cachedir: .pytest_cache
rootdir: f:\git\MLOps-Assignments\HW03\HW3_Student\HW3_Student\HW3_A
plugins: anyio-4.13.0, hydra-core-1.3.2, typeguard-4.5.2
collecting ... collected 23 items

tests/test_adversarial.py::test_missing_bundle_dir PASSED                [  4%]
tests/test_adversarial.py::test_very_long_text PASSED                    [  8%]
tests/test_adversarial.py::test_unicode_text PASSED                      [ 13%]
tests/test_adversarial.py::test_numeric_text PASSED                      [ 17%]
tests/test_adversarial.py::test_single_token_text PASSED                 [ 21%]
tests/test_adversarial.py::test_duplicate_texts PASSED                   [ 26%]
tests/test_adversarial.py::test_batch_of_one PASSED                      [ 30%]
tests/test_adversarial.py::test_large_batch PASSED  

# Step 6: Register in MLflow

Log the bundle to MLflow using `mlflow.sentence_transformers.log_model()`.

Before running this cell, make sure:
1. You have `source .env` (or set the env vars manually)
2. All tests pass
3. `bundle/model/` contains the 6 model files

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
# TODO: Register the bundle in MLflow

# HINT: import mlflow
# HINT: mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
# HINT: mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "qbc12_hw03_encoder"))
#
# HINT: with mlflow.start_run(run_name="hw3a-bundle") as run:
# HINT:     mlflow.log_param("embedding_dim", 384)
# HINT:     mlflow.log_param("max_seq_len", 256)
# HINT:     mlflow.log_param("model_id", MODEL_ID)
# HINT:     mlflow.set_tag("stage", "candidate")
# HINT:
# HINT:     # Log the model
# HINT:     # Option A: sentence_transformers.log_model(...)
# HINT:     model_info = mlflow.sentence_transformers.log_model(
# HINT:         model_source=str(BUNDLE_MODEL_DIR.resolve()),
# HINT:         artifact_path="bundle",
# HINT:         task="llm/v1/embeddings",
# HINT:     )
# HINT:     print("Model URI:", model_info.model_uri)
# HINT:     print("Run ID:", run.info.run_id)

# --- YOUR CODE HERE ---

import mlflow
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "qbc12_hw03_encoder"))
#
with mlflow.start_run(run_name="hw3a-bundle") as run:
    mlflow.log_param("embedding_dim", 384)
    mlflow.log_param("max_seq_len", 256)
    mlflow.log_param("model_id", MODEL_ID)
    mlflow.set_tag("stage", "candidate")

# --- END YOUR CODE ---

print("Done. Check MLflow UI for your registered model.")

2026/06/19 17:02:26 INFO mlflow.tracking.fluent: Experiment with name 'qbc12_hw03_encoder_student_melika_nobakhtian' does not exist. Creating a new experiment.


🏃 View run hw3a-bundle at: http://185.50.38.163:33014/#/experiments/55/runs/a32c83dd604e40d594e8d4d15f949d27
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/55
Done. Check MLflow UI for your registered model.


# Step 7: Upload to MinIO

Upload your `bundle/` directory to the shared MinIO bucket.

Run the provided upload script:
```bash
source .env && bash scripts/01_upload_to_minio.sh
```

Or from this notebook:

In [ ]:
# Upload bundle to MinIO using the provided script
# Make sure .env is sourced first!
!bash scripts/01_upload_to_minio.sh

# Take a screenshot of the upload confirmation for EVIDENCE/minio_upload.png
# *** I ran script using CLI
# **Attention: I replaced the previous command with the following because Minio did not recognize `exclude` flag.

## Submission Checklist

Before submitting, verify:

- [ ] `encoder_bundle.ipynb` — all cells executed with visible outputs
- [ ] `bundle/predict.py` — 4 functions implemented
- [ ] `bundle/metadata.json` — all fields filled (no TODO placeholders)
- [ ] `bundle/MANIFEST.json` — real SHA-256 hashes for every file
- [ ] `bundle/requirements.txt` — pinned dependencies listed
- [ ] `bundle/model/` — 6 model files present
- [ ] `EVIDENCE/pytest_pass.png` — all tests green
- [ ] `EVIDENCE/mlflow_registered.png` — MLflow UI showing the model
- [ ] `EVIDENCE/minio_upload.png` — MinIO upload confirmation

Good luck!